# Notebook 2 — QLoRA Fine-tuning

Run this on **Google Colab** (free T4 GPU) or Kaggle.

**Runtime:** ~2 hours for 1000 examples, 3 epochs on T4.

**Output:** `iam-policy-adapter/` (LoRA weights, ~50MB)

In [1]:
# Run once to install dependencies
!pip install unsloth trl datasets peft accelerate bitsandbytes -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/

## 0. Colab setup — upload training data

Upload `data/processed/train.jsonl` from your local machine to Colab, or mount Google Drive if you stored it there.

In [4]:
import os

if 'google.colab' in str(get_ipython()):
    # Option A — mount Google Drive (recommended if you saved data there)
    # from google.colab import drive
    # drive.mount('/content/drive')
    # DRIVE_PATH = '/content/drive/MyDrive/iam-policy-llm/data/processed'
    os.makedirs('data/processed', exist_ok=True)
    # import shutil
    # shutil.copy(f'{DRIVE_PATH}/train.jsonl', 'data/processed/train.jsonl')
    # print('Loaded train.jsonl from Google Drive')

    # Option B — direct file upload (uncomment if not using Drive)
    from google.colab import files
    os.makedirs('data/processed', exist_ok=True)
    files.upload()  # upload train.jsonl, then move it:
    import shutil
    shutil.move('train.jsonl', 'data/processed/train.jsonl')
    print('Loaded train.jsonl from direct upload')
else:
    assert os.path.exists('data/processed/train.jsonl'), \
        'Run notebook 01 first to generate data/processed/train.jsonl'
    print('Local run — data/processed/train.jsonl found')


Saving train.jsonl to train.jsonl
Loaded train.jsonl from direct upload


In [3]:
print(f'Listing files in {DRIVE_PATH}:')
!ls -F "{DRIVE_PATH}"

Listing files in /content/drive/MyDrive/iam-policy-llm/data/processed:
ls: cannot access '/content/drive/MyDrive/iam-policy-llm/data/processed': No such file or directory


## 1. Load base model in 4-bit (QLoRA)

In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/llama-3.2-3b-instruct',
    max_seq_length = 2048,
    load_in_4bit = True,
)
print('Model loaded')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model loaded


## 2. Attach LoRA adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = 'none',
)
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.5 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 9,175,040 || all params: 3,221,924,864 || trainable%: 0.2848


## 3. Load and format dataset

In [7]:
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def format_prompt(ex):
    return (
        f"### Instruction:\n{ex['instruction']}\n\n"
        f"### Input:\n{ex['input']}\n\n"
        f"### Response:\n{json.dumps(ex['output'], indent=2)}"
    )

raw = load_jsonl('data/processed/train.jsonl')
dataset = Dataset.from_list([{'text': format_prompt(ex)} for ex in raw])
print(f'{len(dataset)} training examples')
print(dataset[0]['text'][:500])

451 training examples
### Instruction:
Generate an AWS IAM policy for the following requirement

### Input:
A web server role needs to list and get ACM certificates, but is explicitly forbidden from deleting or modifying any certificates.

### Response:
{
  "policy": {
    "Version": "2012-10-17",
    "Statement": [
      {
        "Effect": "Allow",
        "Action": [
          "acm:ListCertificates",
          "acm:GetCertificate",
          "acm:DescribeCertificate"
        ],
        "Resource": "arn:aws:acm:us-


## 4. Train

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = 'text',
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = True,
        output_dir = 'outputs',
        logging_steps = 10,
        save_steps = 100,
    ),
)
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/451 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 451 | Num Epochs = 3 | Total steps = 171
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.234537
20,0.788441
30,0.699694
40,0.671672
50,0.633384
60,0.631166
70,0.565498
80,0.578116
90,0.558394
100,0.547918


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-171/tokenizer_config.json.


TrainOutput(global_step=171, training_loss=0.6167816700991134, metrics={'train_runtime': 474.5007, 'train_samples_per_second': 2.851, 'train_steps_per_second': 0.36, 'total_flos': 6289067599128576.0, 'train_loss': 0.6167816700991134, 'epoch': 3.0})

## 7. (Optional) Push adapter to HuggingFace Hub

Skip this if you only want local CPU inference — the adapter from step 6 is enough.

Useful if you want to share the model or deploy to HuggingFace Spaces.  
Requires a free HuggingFace account and a write token (`huggingface-cli login` or `HF_TOKEN` env var).

In [ ]:
from huggingface_hub import HfApi

HF_REPO = 'YOUR_HF_USERNAME/llama-3.2-3b-iam-policy'  # <-- change this

api = HfApi()
api.create_repo(HF_REPO, repo_type='model', exist_ok=True)

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f'Adapter pushed to https://huggingface.co/{HF_REPO}')
print('Update HF_REPO in notebook 04 to point to this model.')

## 5. Save adapter locally

Saves the LoRA weights (~50 MB) to `iam-policy-adapter/` inside the Colab session.

In [9]:
model.save_pretrained('iam-policy-adapter')
tokenizer.save_pretrained('iam-policy-adapter')
print('Adapter saved to iam-policy-adapter/ — proceed to step 6 to download it.')

Unsloth: Restored added_tokens_decoder metadata in iam-policy-adapter/tokenizer_config.json.


Adapter saved to iam-policy-adapter/ — proceed to step 6 to download it.


## 6. Download adapter to your PC

**Option A — Google Drive (recommended)**  
Copies the adapter to your Drive so you can download it from drive.google.com.

**Option B — Direct Colab download**  
Zips the adapter and triggers a browser download.

In [10]:
import os, shutil

# ── Option A: save to Google Drive ──────────────────────────────────────────
# After this cell, open drive.google.com and download iam-policy-llm/iam-policy-adapter/

if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')                          # skip if already mounted

    dest = '/content/drive/MyDrive/iam-policy-llm/iam-policy-adapter'
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree('iam-policy-adapter', dest)
    print(f'Adapter copied to Google Drive: {dest}')
    print('Download it from drive.google.com → MyDrive/iam-policy-llm/')

# ── Option B: direct browser download (zip) ─────────────────────────────────
# Uncomment the lines below if you prefer a direct download instead of Drive.

# from google.colab import files
# shutil.make_archive('iam-policy-adapter', 'zip', '.', 'iam-policy-adapter')
# files.download('iam-policy-adapter.zip')   # triggers browser download (~50 MB)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adapter copied to Google Drive: /content/drive/MyDrive/iam-policy-llm/iam-policy-adapter
Download it from drive.google.com → MyDrive/iam-policy-llm/
